In [1]:
import pypsa
from pypsa.components._types import buses

import fbmc  # noqa: F401  not unused; registers pypsa.result.zonal_net.fbmc accessor
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import networkx as nx


In [45]:
nodal_net = pypsa.Network("inputs/test/base_s_25_elec_Ep100.nc")

INFO:pypsa.network.io:New version 1.2.4 available! (Current: 0.35.1)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, lines, links, loads, storage_units


In [31]:

nodal_net.buses["zone_name"] = nodal_net.buses.country
#nodal_net.buses.rename(columns={"country": "zone_name"}, inplace=True)
nodal_net.buses.loc["DK1 0", "zone_name"] = "DK-10"
nodal_net.buses.loc["DK1 3", "zone_name"] = "DK-13"
nodal_net.buses.loc["DK3 0", "zone_name"] = "DK-30"
nodal_net.buses.loc["GB2 0", "zone_name"] = "NIR"
nodal_net.remove("Bus", "DK1 3")


In [47]:
print(f" baseline network lines: {nodal_net.lines.loc[['1', '2'], 's_nom_opt']}")

 baseline network lines: Line
1     1005.456135
2    20547.542335
Name: s_nom_opt, dtype: float64


In [42]:
nodal_net.lines.columns

Index(['bus0', 'bus1', 'type', 'x', 'r', 'g', 'b', 's_nom', 's_nom_mod',
       's_nom_extendable', 's_nom_min', 's_nom_max', 's_max_pu',
       'capital_cost', 'active', 'build_year', 'lifetime', 'length', 'carrier',
       'terrain_factor', 'num_parallel', 'v_ang_min', 'v_ang_max',
       'sub_network', 'x_pu', 'r_pu', 'g_pu', 'b_pu', 'x_pu_eff', 'r_pu_eff',
       's_nom_opt', 'v_nom', 'i_nom', 'dc'],
      dtype='object')

In [43]:
df = nodal_net.lines
df.loc[lambda df: (
    (df["bus0"].str.contains("FR") & df["bus1"].str.contains("BE"))
    | (df["bus0"].str.contains("BE") & df["bus1"].str.contains("FR"))
)]

,bus0,bus1,type,x,r,g,b,s_nom,s_nom_mod,s_nom_extendable,...,x_pu,r_pu,g_pu,b_pu,x_pu_eff,r_pu_eff,s_nom_opt,v_nom,i_nom,dc
Line,,,,,,,,,,,,,,,,,,,,,
1,BE1 0,FR1 2,Al/St 240/40 4-bundle 380.0,115.875914,14.131209,0.0,0.000716,1005.455494,0.0,True,...,0.000802,0.000098,0.0,0.0,0.000802,0.000098,1005.456135,380.0,2.58,0.0
2,BE1 0,FR1 6,Al/St 240/40 4-bundle 380.0,10.245231,1.249418,0.0,0.002154,5865.157047,0.0,True,...,0.000071,0.000009,0.0,0.0,0.000071,0.000009,20547.542335,380.0,2.58,0.0


In [13]:
lambda df: df["carrier"] == "solar"(nodal_net.generators)

<>:1: SyntaxWarning: 'str' object is not callable; perhaps you missed a comma?
<>:1: SyntaxWarning: 'str' object is not callable; perhaps you missed a comma?
/var/folders/cn/l4_9msnj2pb5tkz_l18sbgdh0000gn/T/ipykernel_77032/855659233.py:1: SyntaxWarning: 'str' object is not callable; perhaps you missed a comma?
  lambda df: df["carrier"] == "solar"(nodal_net.generators)


<function __main__.<lambda>(df)>

In [11]:
lambda df: df.index == "BE1 0 nuclear"

<>:1: SyntaxWarning: 'str' object is not callable; perhaps you missed a comma?
<>:1: SyntaxWarning: 'str' object is not callable; perhaps you missed a comma?
/var/folders/cn/l4_9msnj2pb5tkz_l18sbgdh0000gn/T/ipykernel_77032/1943451384.py:1: SyntaxWarning: 'str' object is not callable; perhaps you missed a comma?
  lambda df: df.index == "BE1 0 nuclear"(nodal_net.generators)


<function __main__.<lambda>(df)>

In [47]:
import copy
import pandas as pd
import pypsa

def _resolve_mask(index, selector, static_df):
    """Return a boolean Series aligned to `index` marking selected assets."""
    if selector == "all":
        return pd.Series(True, index=index)
    if callable(selector):
        mask = selector(static_df)
        return mask.reindex(index, fill_value=False)
    # assume list/iterable of names
    return pd.Series(index.isin(selector), index=index)

def apply_perturbation(network: pypsa.Network, param: dict, direction: str) -> pypsa.Network:
    """
    Return a COPY of `network` with a single parameter perturbed.

    direction: 'low' -> value * (1 - variation)
               'high' -> value * (1 + variation)
    """
    n = copy.deepcopy(network)
    component = param["component"]
    attribute = param["attribute"]
    factor = (1 - param["variation"]) if direction == "low" else (1 + param["variation"])

    static_df = getattr(n, component)
    time_container = getattr(n, f"{component}_t", None)

    if time_container is not None and attribute in time_container:
        ts = time_container[attribute]  # DataFrame: snapshots x assets
        mask = _resolve_mask(ts.columns, param["selector"], static_df)
        cols = ts.columns[mask.values]
        ts.loc[:, cols] = ts.loc[:, cols] * factor
        print((ts.loc[:, cols] * factor).head())

    elif attribute in static_df.columns:
        mask = _resolve_mask(static_df.index, param["selector"], static_df)
        static_df.loc[mask, attribute] = static_df.loc[mask, attribute] * factor

    else:
        raise ValueError(
            f"Attribute '{attribute}' not found as a static column on "
            f"n.{component} nor as a time series on n.{component}_t"
        )

    return n

parameters =    {
        "name": "Renewable production",
        "component": "generators",
        "attribute": "p_max_pu",
        "selector": lambda df: df["carrier"].isin(["onwind", "solar", "solar-hsat"]),
        "variation": 0.20,
    }

direction = "low"

nodal_net_pert = apply_perturbation(nodal_net, parameters, direction)


Generator            BE1 0 0 solar  DE1 0 0 solar  DE1 1 0 solar  \
snapshot                                                           
2013-01-01 00:00:00            0.0            0.0            0.0   
2013-01-01 01:00:00            0.0            0.0            0.0   
2013-01-01 02:00:00            0.0            0.0            0.0   
2013-01-01 03:00:00            0.0            0.0            0.0   
2013-01-01 04:00:00            0.0            0.0            0.0   

Generator            DE1 2 0 solar  DE1 3 0 solar  DE1 4 0 solar  \
snapshot                                                           
2013-01-01 00:00:00            0.0            0.0            0.0   
2013-01-01 01:00:00            0.0            0.0            0.0   
2013-01-01 02:00:00            0.0            0.0            0.0   
2013-01-01 03:00:00            0.0            0.0            0.0   
2013-01-01 04:00:00            0.0            0.0            0.0   

Generator            DE1 5 0 solar  DE1 6 0 so

In [50]:
nodal_net_pert.generators_t.p_max_pu.loc[:, "NL1 0 0 onwind"]

snapshot
2013-01-01 00:00:00    0.798755
2013-01-01 01:00:00    0.791360
2013-01-01 02:00:00    0.771815
2013-01-01 03:00:00    0.705320
2013-01-01 04:00:00    0.549519
                         ...   
2013-12-30 19:00:00    0.794926
2013-12-30 20:00:00    0.787824
2013-12-30 21:00:00    0.761298
2013-12-30 22:00:00    0.684462
2013-12-30 23:00:00    0.643835
Name: NL1 0 0 onwind, Length: 8736, dtype: float64

In [33]:
zone_iteration

'DE1 0'

In [34]:
nodal_net.generators[nodal_net.generators.carrier.isin(["CCGT"])].query('bus == @zone_iteration').p_nom

Generator
DE1 0 CCGT    2231.3126
Name: p_nom, dtype: float64

In [5]:
nodal_net.generators.carrier.unique()

array(['CCGT', 'biomass', 'nuclear', 'oil', 'waste', 'OCGT', 'coal',
       'geothermal', 'lignite', 'offwind-dc', 'solar', 'offwind-ac',
       'offwind-float', 'solar-hsat', 'onwind'], dtype=object)

In [87]:
def freeze_expansion(network):
    """
    Replace nominal capacities by their optimized values and
    disable further expansion.

    Parameters
    ----------
    network : pypsa.Network
    """

    for comp in network.components.values():
        df = getattr(network, comp.list_name, None)

        if df is None or df.empty:
            continue

        cols = df.columns

        # p_nom -> p_nom_opt
        if "p_nom" in cols and "p_nom_opt" in cols:
            df["p_nom"] = df["p_nom_opt"]
            if "p_nom_extendable" in cols:
                df["p_nom_extendable"] = False

        # s_nom -> s_nom_opt
        if "s_nom" in cols and "s_nom_opt" in cols:
            df["s_nom"] = df["s_nom_opt"]
            if "s_nom_extendable" in cols:
                df["s_nom_extendable"] = False

        # e_nom -> e_nom_opt
        if "e_nom" in cols and "e_nom_opt" in cols:
            df["e_nom"] = df["e_nom_opt"]
            if "e_nom_extendable" in cols:
                df["e_nom_extendable"] = False

    return network

In [88]:
freeze_expansion(nodal_net)

PyPSA Network 'Unnamed Network'
-------------------------------
Components:
 - Bus: 25
 - Carrier: 19
 - Generator: 256
 - Line: 46
 - Link: 36
 - Load: 25
 - StorageUnit: 25
Snapshots: 8736

In [11]:
bus_zone_map = nodal_net.buses['zone_name']
zonal_net = nodal_net.fbmc.to_zonal(bus_zone_map)

/Users/ng-work/Models/pypsa-fbmc/src/fbmc/input_network_conversions/network_conversion.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bus_data_to_transfer.loc[:, 'country'] = oldnet.buses['country']


In [60]:
components = {
    "producer_surplus": "a",
    "storage_surplus": "a",
    "consumer_surplus": "a",
    "line_congestion_rent": "a",
    "link_congestion_rent": "a",
}

In [90]:
zonal_net.set_snapshots(zonal_net.snapshots[:24])

In [76]:
zonal_net.lines.head()

,bus0,bus1,type,x,r,g,b,s_nom,s_nom_mod,s_nom_extendable,...,v_ang_min,v_ang_max,sub_network,x_pu,r_pu,g_pu,b_pu,x_pu_eff,r_pu_eff,s_nom_opt
Line,,,,,,,,,,,,,,,,,,,,,


In [91]:
import yaml

with open("inputs/test/config.base_s_25_elec_Ep100.yaml") as stream:
    try:
        config_original = (yaml.safe_load(stream))
    except yaml.YAMLError as exc:
        print(exc)

solver_options_net = config_original["solving"]["solver_options"]["gurobi-default"]
solver_name = config_original["solving"][("solver")]
solver_name = {'solver_name': "gurobi"}
solver_options = solver_name | solver_options_net
solver_options

{'solver_name': 'gurobi',
 'threads': 32,
 'method': 2,
 'crossover': 0,
 'BarConvTol': 1e-05,
 'Seed': 123,
 'AggFill': 0,
 'PreDual': 0,
 'GURO_PAR_BARDENSETHRESH': 200}

In [92]:
config = fbmc.FBMCConfig(
    base_case_strategy=fbmc.BaseCaseStrategy.ZERO_FLOWS,
    add_security_constraints=False,
    gsk_strategy=fbmc.GSKStrategy.P_NOM,
    security_constraint_bodf_size_threshold=0.1,
    min_ram=0.1,
    reliability_margin_factor=0.0,
    advanced_hybrid_coupling_flag= True,
    #solver_kwargs={'solver_name': 'gurobi', 'OutputFlag': 0},
    solver_kwargs=solver_options
)

zonal_net.fbmc.create_model(nodal_net, config=config)

       'relation/15772117-320-DC', 'relation/15781671-525-DC',
       'relation/2127794-270-DC', 'relation/2505320-400-DC',
       'relation/5487095-400-DC', 'relation/6914309-500-DC',
       'relation/8184641-200-DC', 'relation/8185420-320-DC',
       'relation/8185487-400-DC', 'relation/8193755-320-DC', 'TYNDP2020_17',
       'TYNDP2020_32', 'TYNDP2020_36', 'DC2', 'TYNDP2024_153', 'TYNDP2024_285',
       'TYNDP2022_286'],
      dtype='object', name='Link')
INFO:root:Determined 4 sub-networks in the base case nodal network.
INFO:root:Created optimization model without meshed split.


Fixed load has coordinate Bus


Linopy LP model

Variables:
----------
 * Generator-p (snapshot, Generator)
 * Link-p (snapshot, Link)
 * StorageUnit-p_dispatch (snapshot, StorageUnit)
 * StorageUnit-p_store (snapshot, StorageUnit)
 * StorageUnit-state_of_charge (snapshot, StorageUnit)
 * Zone-p (snapshot, Zone)

Constraints:
------------
 * Generator-fix-p-lower (snapshot, Generator-fix)
 * Generator-fix-p-upper (snapshot, Generator-fix)
 * Link-fix-p-lower (snapshot, Link-fix)
 * Link-fix-p-upper (snapshot, Link-fix)
 * StorageUnit-fix-p_dispatch-lower (snapshot, StorageUnit-fix)
 * StorageUnit-fix-p_dispatch-upper (snapshot, StorageUnit-fix)
 * StorageUnit-fix-p_store-lower (snapshot, StorageUnit-fix)
 * StorageUnit-fix-p_store-upper (snapshot, StorageUnit-fix)
 * StorageUnit-fix-state_of_charge-lower (snapshot, StorageUnit-fix)
 * StorageUnit-fix-state_of_charge-upper (snapshot, StorageUnit-fix)
 * StorageUnit-energy_balance (snapshot, StorageUnit)
 * Zone-definition (snapshot, Zone)
 * Bus-nodal_balance (Bus, sn

In [93]:
zonal_net.model.solve(**config.solver_kwargs)
result = zonal_net.fbmc.results()
print(result)

INFO:linopy.model: Solve problem using Gurobi solver
INFO:linopy.model:Solver options:
 - threads: 32
 - method: 2
 - crossover: 0
 - BarConvTol: 1e-05
 - Seed: 123
 - AggFill: 0
 - PreDual: 0
 - GURO_PAR_BARDENSETHRESH: 200
INFO:linopy.io: Writing time: 0.05s


Set parameter Username


INFO:gurobipy:Set parameter Username


Set parameter LicenseID to value 2838876


INFO:gurobipy:Set parameter LicenseID to value 2838876


...

Optimal objective 1.21855602e+07


INFO:gurobipy:Optimal objective 1.21855602e+07


INFO:gurobipy:
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 8640 primals, 19872 duals
Objective: 1.22e+07
Solver model: available
Solver message: 2



FBMCResult
  zonal_net: Unnamed Network | snapshots=24 | components=[Bus:10, Generator:256, Load:25, Link:19, StorageUnit:25]
  base_case: Unnamed Network | snapshots=8736 | components=[Bus:25, Generator:256, Load:25, Line:46, Link:36, StorageUnit:25, SubNetwork:4]
  net_positions: DataFrame(24, 10)
  dispatch_results: DispatchResult object with attrs: 
  generators_p: (24, 256) snapshots x generators, 
  storage_units_p: (24, 25) snapshots x storage units, 
  links_p0: (24, 19) snapshots x links, 
  storage_levels: (24, 25) snapshots x storage levels, 
  water_values: (24, 25) snapshots x water values
  fbmc_parameters: 2 subnet(s) [0, 2]


/Users/ng-work/.pyenv/versions/pypsa-fbmc/lib/python3.11/site-packages/linopy/common.py:492: UserWarning: Coordinates across variables not equal. Perform outer join.
  warn(
/Users/ng-work/.pyenv/versions/pypsa-fbmc/lib/python3.11/site-packages/linopy/common.py:492: UserWarning: Coordinates across variables not equal. Perform outer join.
  warn(


In [97]:
def get_prices(zonal_net, nodal_net):
    ## prices per zone from the netposition
    for zone in zonal_net.buses.index.unique():
        zonal_net.buses_t.marginal_price[zone] = (
                zonal_net.model.constraints["Zone-definition"].loc[:, zone].dual.to_pandas() * -1)

    ##prices per zone from the nodal prices
    ### defining buses with nodal prices
    # Assuming your network is called n
    nodal_net.determine_network_topology()
    # Count buses in each subnetwork
    subnet_sizes = nodal_net.buses.groupby("sub_network").size()
    # Subnetworks with fewer than 3 buses
    small_subnets = subnet_sizes[subnet_sizes < 3].index
    # Bus names in those subnetworks
    small_subnet_buses = nodal_net.buses[nodal_net.buses["sub_network"].isin(small_subnets)].index.tolist()
    small_subnet_zones = nodal_net.buses.zone_name.loc[small_subnet_buses].values

    for zone in small_subnet_zones:
        zonal_net.buses_t.marginal_price[zone] = zonal_net.model.constraints["Bus-nodal_balance"].loc[
                                                        zone, :].dual.to_pandas()

    return zonal_net
result.zonal_net = get_prices(result.zonal_net, nodal_net)

In [98]:
def get_lines_results(zonal_net, nodal_net, fbmc_parameters, net_positions):
    bus_zone_map = nodal_net.buses['zone_name']

    zonal_net.lines = nodal_net.lines
    zonal_net.lines["zone0"] = zonal_net.lines.bus0.map(bus_zone_map)
    zonal_net.lines["zone1"] = zonal_net.lines.bus1.map(bus_zone_map)

    zonal_net.lines_t.p0 = (fbmc_parameters["0"].z_ptdf * net_positions).sum(
        dim='Zone').to_pandas()

    return zonal_net
result.zonal_net = get_lines_results(result.zonal_net, nodal_net, result.fbmc_parameters, result.net_positions)

In [100]:
zonal_net.lines["zone0"]

Line
0      BE
1      BE
10     DE
11     DE
12     DE
13     DE
14     DE
15     DE
16     DE
17     DE
18     DE
19     DE
2      BE
20     DE
21     DE
22     DE
23     DE
24     DE
25     DE
26     DE
27     DE
28     FR
29     FR
3      BE
30     FR
31     FR
32     FR
33     FR
34     FR
35     FR
36     FR
37     FR
38     FR
39     FR
4      BE
40     GB
41     GB
42     GB
43     GB
44    NIR
45    NIR
5      DE
6      DE
7      DE
8      DE
9      DE
Name: zone0, dtype: object

In [101]:
zonal_net.lines["zone1"]

Line
0      BE
1      BE
10     DE
11     DE
12     DE
13     DE
14     DE
15     DE
16     DE
17     DE
18     DE
19     DE
2      BE
20     DE
21     DE
22     DE
23     DE
24     DE
25     DE
26     DE
27     DE
28     FR
29     FR
3      BE
30     FR
31     FR
32     FR
33     FR
34     FR
35     FR
36     FR
37     FR
38     FR
39     FR
4      BE
40     GB
41     GB
42     GB
43     GB
44    NIR
45    NIR
5      DE
6      DE
7      DE
8      DE
9      DE
Name: zone1, dtype: object

In [81]:
lines = result.zonal_net.lines
price0 = result.zonal_net.buses_t.marginal_price
#[lines["zone0"]]

In [107]:
for key in result.fbmc_parameters.keys():
    print(key)

0
2


In [123]:
z_ptdf = xr.concat(
    [ds.z_ptdf for ds in result.fbmc_parameters.values()],
    dim="cnec"
)

(
    z_ptdf * result.net_positions
).sum(dim="Zone")

<xarray.DataArray (snapshot: 24, cnec: 44)> Size: 8kB
array([[-5.30613688e+03,  4.94033581e+02, -3.29159675e+03, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-5.21399757e+03,  5.24152207e+02, -3.40157419e+03, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-5.17573499e+03,  5.42990833e+02, -2.76252296e+03, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       ...,
       [-1.56619315e+03, -8.49371933e-01, -2.38389148e+03, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-2.40805654e+03,  1.16210767e+02, -2.25659501e+03, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-2.78008251e+03,  1.78533805e+02, -2.15984627e+03, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00]])
Coordinates:
  * snapshot          (snapshot) datetime64[ns] 192B 2013-01-01 ... 2013-01-0...
  * cnec              (cnec) object 352B '0' '1' '10' '11' ... '41' '42' '43'
    branch            (cnec) object 352B '0' '1' '10' '11' ... '41' '42' '43'
    branch_component  (cnec) object 352B 'Line' 'Line' 'Line' ... 'Line' 'Line'

In [121]:
res_0 = (result.fbmc_parameters["0"].z_ptdf * result.net_positions).sum(dim='Zone')
res_2 = (result.fbmc_parameters["2"].z_ptdf * result.net_positions).sum(dim='Zone')
xr.concat([res_0, res_2], dim="cnec")

<xarray.DataArray (snapshot: 24, cnec: 44)> Size: 8kB
array([[-5.30613688e+03,  4.94033581e+02, -3.29159675e+03, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-5.21399757e+03,  5.24152207e+02, -3.40157419e+03, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-5.17573499e+03,  5.42990833e+02, -2.76252296e+03, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       ...,
       [-1.56619315e+03, -8.49371933e-01, -2.38389148e+03, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-2.40805654e+03,  1.16210767e+02, -2.25659501e+03, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-2.78008251e+03,  1.78533805e+02, -2.15984627e+03, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00]])
Coordinates:
  * snapshot          (snapshot) datetime64[ns] 192B 2013-01-01 ... 2013-01-0...
  * cnec              (cnec) object 352B '0' '1' '10' '11' ... '41' '42' '43'
    branch            (cnec) object 352B '0' '1' '10' '11' ... '41' '42' '43'
    branch_component  (cnec) object 352B 'Line' 'Line' 'Line' ... 'Line' 'Line'

In [110]:
result.fbmc_parameters["2"].z_ptdf

<xarray.DataArray (snapshot: 8736, Zone: 1, cnec: 4)> Size: 280kB
array([[[-0.38173559, -0.32867042,  0.10642649, -0.1619839 ]],

       [[-0.38173559, -0.32867042,  0.10642649, -0.1619839 ]],

       [[-0.38173559, -0.32867042,  0.10642649, -0.1619839 ]],

       ...,

       [[-0.38173559, -0.32867042,  0.10642649, -0.1619839 ]],

       [[-0.38173559, -0.32867042,  0.10642649, -0.1619839 ]],

       [[-0.38173559, -0.32867042,  0.10642649, -0.1619839 ]]])
Coordinates:
  * snapshot          (snapshot) datetime64[ns] 70kB 2013-01-01 ... 2013-12-3...
  * Zone              (Zone) object 8B 'GB'
    branch            (cnec) object 32B '40' '41' '42' '43'
    branch_component  (cnec) object 32B 'Line' 'Line' 'Line' 'Line'
  * cnec              (cnec) object 32B '40' '41' '42' '43'

In [201]:
lines_zone_map = nodal_net.lines[["bus0", "bus1"]].replace(bus_zone_map)
lines_cb_zone_map = lines_zone_map[lines_zone_map.nunique(axis=1) > 1]
lines_cb_zone_map


,bus0,bus1
Line,,
0,BE,DE
1,BE,FR
10,DE,DK-10
11,DE,DK-10
2,BE,FR
20,DE,FR
21,DE,LU
24,DE,LU
25,DE,NL


In [202]:
line = "0"

In [207]:
bus0 = lines_cb_zone_map.loc["0", "bus0"]
bus1 = lines_cb_zone_map.loc["0", "bus1"]
p0 = result.zonal_net.buses_t.marginal_price.loc[:,bus0]
p1 = result.zonal_net.buses_t.marginal_price.loc[:,bus1]
diff = p0-p1

In [216]:
np.abs((commercial_flows.to_pandas()[line] * diff).sum()) #this is the congestion rent earned annually on that line

6732879.853592924

In [193]:
(result.net_positions * result.zonal_net.buses_t.marginal_price).to_pandas()

Zone,BE,DE,DK-10,DK-30,FR,GB,IE,LU,NIR,NL
snapshot,,,,,,,,,,
2013-01-01 00:00:00,-25363.830322,681.525322,501.799400,2.187603e-07,-780184.668648,-0.0,4.632971e-07,-15.123860,0.000011,42.404729
2013-01-01 01:00:00,-19602.264339,657.164866,644.283991,2.185129e-07,-788316.878010,-0.0,4.585219e-07,-17.441266,0.000022,53.098302
2013-01-01 02:00:00,-16097.812774,717.573053,1143.329424,2.183400e-07,-795097.223833,-0.0,4.581352e-07,-13.539947,0.000073,60.970260
2013-01-01 03:00:00,-14578.679452,761.003901,964.339931,2.182327e-07,-798028.416253,-0.0,4.575499e-07,-12.916389,0.000073,64.579668
2013-01-01 04:00:00,-15516.382640,802.426062,860.996513,2.189778e-07,-797644.193603,-0.0,4.570239e-07,-12.613065,0.000073,89.390906
...,...,...,...,...,...,...,...,...,...,...
2013-03-25 03:00:00,-93579.929839,388766.078451,199309.389244,4.298364e-04,-662881.162166,-0.0,3.592514e-04,-8351.820747,0.000338,179798.454784
2013-03-25 04:00:00,-105625.694772,300434.045745,204444.097341,2.456213e-04,-554580.133521,-0.0,3.592453e-04,-8803.812992,0.000338,167358.901416
2013-03-25 05:00:00,-132545.630208,185951.387592,223126.730876,2.456220e-04,-396899.534176,-0.0,3.592450e-04,-9350.093666,0.000338,133145.255756


In [164]:
nodal_net.lines.head()

,bus0,bus1,type,x,r,g,b,s_nom,s_nom_mod,s_nom_extendable,...,x_pu,r_pu,g_pu,b_pu,x_pu_eff,r_pu_eff,s_nom_opt,v_nom,i_nom,dc
Line,,,,,,,,,,,,,,,,,,,,,
0,BE1 0,DE1 2,Al/St 240/40 4-bundle 380.0,31.843002,3.883293,0.0,0.002245,4853.665269,0.0,False,...,0.000221,0.000027,0.0,324.142712,0.000221,0.000027,4853.665269,380.0,2.58,NaN
1,BE1 0,FR1 2,Al/St 240/40 4-bundle 380.0,115.875914,14.131209,0.0,0.000716,1005.456135,0.0,False,...,0.000802,0.000098,0.0,103.383697,0.000802,0.000098,1005.456135,380.0,2.58,0.0
10,DE1 0,DK1 0,Al/St 240/40 4-bundle 380.0,32.118103,3.916842,0.0,0.002264,4386.258887,0.0,False,...,0.000222,0.000027,0.0,326.943074,0.000222,0.000027,4386.258887,380.0,2.58,NaN
11,DE1 0,DK1 0,Al/St 240/40 4-bundle 380.0,32.118103,3.916842,0.0,0.002264,4386.258092,0.0,False,...,0.000222,0.000027,0.0,326.943074,0.000222,0.000027,4386.258092,380.0,2.58,NaN
12,DE1 1,DE1 3,Al/St 240/40 4-bundle 380.0,24.568469,2.996155,0.0,0.006928,6792.450481,0.0,False,...,0.000170,0.000021,0.0,1000.366455,0.000170,0.000021,6792.450481,380.0,2.58,0.0


In [23]:
result.zonal_net.links_t.p0.head()

Link,relation/10377412-320-DC,relation/14126301-450-DC,relation/15772117-320-DC,relation/15781671-525-DC,relation/2127794-270-DC,relation/2505320-400-DC,relation/5487095-400-DC,relation/6914309-500-DC,relation/8184641-200-DC,relation/8185420-320-DC,relation/8185487-400-DC,relation/8193755-320-DC,TYNDP2020_17,TYNDP2020_32,TYNDP2020_36,DC2,TYNDP2024_153,TYNDP2024_285,TYNDP2022_286
snapshot,,,,,,,,,,,,,,,,,,,
2013-01-01 00:00:00,-1000.012677,-996.568035,1000.022479,-702.993219,2000.010709,-594.456523,-597.490260,-470.017348,-137.683077,-697.313181,1000.002192,-1000.007636,0.259695,0.177802,0.577336,0.629108,0.455370,0.692529,0.172865
2013-01-01 01:00:00,-1000.012590,-998.042808,1000.022393,1349.428652,2000.010620,-594.530931,-598.170140,-439.787309,-342.012694,-697.848433,1000.002039,-1000.007797,0.259697,0.173465,0.577338,0.629107,0.455372,0.692532,0.173117
2013-01-01 02:00:00,-1000.012570,-995.838324,1000.022372,1398.807930,2000.010616,-594.135012,-599.330664,-494.292038,-196.361951,-699.405375,1000.002129,-1000.008067,0.259698,0.181242,0.577338,0.629101,0.455372,0.692532,0.172956
2013-01-01 03:00:00,-1000.012581,-995.151147,1000.022384,1398.919242,2000.010650,-593.929647,-599.351556,-494.794589,-232.742873,-699.421396,1000.002061,-1000.008011,0.259698,0.181029,0.577336,0.629102,0.455370,0.692530,0.173288
2013-01-01 04:00:00,-1000.012612,912.965331,1000.022414,1398.984452,2000.010703,-592.036586,-599.393501,-494.912031,-425.367893,-699.050003,1000.001833,-1000.008027,0.259698,0.181150,0.577333,0.629100,0.455367,0.692527,0.174286


In [33]:
result.zonal_net.lines = nodal_net.lines

In [ ]:
result.zonal_net.lines.

In [36]:
result.zonal_net.lines_t.p0 = (result.fbmc_parameters["0"].z_ptdf * result.net_positions).sum(dim='Zone').to_pandas()

In [38]:
result.zonal_net.links_t.p0

Link,relation/10377412-320-DC,relation/14126301-450-DC,relation/15772117-320-DC,relation/15781671-525-DC,relation/2127794-270-DC,relation/2505320-400-DC,relation/5487095-400-DC,relation/6914309-500-DC,relation/8184641-200-DC,relation/8185420-320-DC,relation/8185487-400-DC,relation/8193755-320-DC,TYNDP2020_17,TYNDP2020_32,TYNDP2020_36,DC2,TYNDP2024_153,TYNDP2024_285,TYNDP2022_286
snapshot,,,,,,,,,,,,,,,,,,,
2013-01-01 00:00:00,-1000.012677,-996.568035,1000.022479,-702.993219,2000.010709,-594.456523,-597.490260,-470.017348,-137.683077,-697.313181,1000.002192,-1000.007636,0.259695,0.177802,0.577336,0.629108,0.455370,0.692529,0.172865
2013-01-01 01:00:00,-1000.012590,-998.042808,1000.022393,1349.428652,2000.010620,-594.530931,-598.170140,-439.787309,-342.012694,-697.848433,1000.002039,-1000.007797,0.259697,0.173465,0.577338,0.629107,0.455372,0.692532,0.173117
2013-01-01 02:00:00,-1000.012570,-995.838324,1000.022372,1398.807930,2000.010616,-594.135012,-599.330664,-494.292038,-196.361951,-699.405375,1000.002129,-1000.008067,0.259698,0.181242,0.577338,0.629101,0.455372,0.692532,0.172956
2013-01-01 03:00:00,-1000.012581,-995.151147,1000.022384,1398.919242,2000.010650,-593.929647,-599.351556,-494.794589,-232.742873,-699.421396,1000.002061,-1000.008011,0.259698,0.181029,0.577336,0.629102,0.455370,0.692530,0.173288
2013-01-01 04:00:00,-1000.012612,912.965331,1000.022414,1398.984452,2000.010703,-592.036586,-599.393501,-494.912031,-425.367893,-699.050003,1000.001833,-1000.008027,0.259698,0.181150,0.577333,0.629100,0.455367,0.692527,0.174286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2013-03-25 03:00:00,-1000.009171,1000.012243,1000.018962,1399.999790,2000.006507,453.870690,-597.320950,36.866317,-380.545053,-694.420663,999.999853,-995.721961,0.257526,0.364664,0.575288,0.278440,0.453325,0.690477,0.172652
2013-03-25 04:00:00,-1000.009210,1000.011828,1000.019000,1399.999230,2000.006587,421.198296,-597.270132,34.647735,-405.466884,-694.357627,999.999711,-995.609983,0.257510,0.364656,0.575272,0.277507,0.453309,0.690461,0.172906
2013-03-25 05:00:00,-1000.009026,1000.011660,1000.018815,1399.999178,2000.006458,398.817486,-597.291395,41.970843,-385.128714,-694.489907,1000.000447,-995.979982,0.257543,0.364648,0.575297,0.280509,0.453334,0.690487,0.172912


In [47]:
def get_lines_results(zonal_net, nodal_net, fbmc_parameters, net_positions):
    zonal_net.lines = nodal_net.lines
    zonal_net.lines_t.p0 = (fbmc_parameters["0"].z_ptdf * net_positions).sum(
        dim='Zone').to_pandas()

In [48]:
get_lines_results(zonal_net, nodal_net, result.fbmc_parameters, result.net_positions)

In [59]:
result.zonal_net.lines.head()
result.zonal_net.lines["zone0"] = result.zonal_net.lines.bus0.map(bus_zone_map)
result.zonal_net.lines.zone0

Line
0      BE
1      BE
10     DE
11     DE
12     DE
13     DE
14     DE
15     DE
16     DE
17     DE
18     DE
19     DE
2      BE
20     DE
21     DE
22     DE
23     DE
24     DE
25     DE
26     DE
27     DE
28     FR
29     FR
3      BE
30     FR
31     FR
32     FR
33     FR
34     FR
35     FR
36     FR
37     FR
38     FR
39     FR
4      BE
40     GB
41     GB
42     GB
43     GB
44    NIR
45    NIR
5      DE
6      DE
7      DE
8      DE
9      DE
Name: zone0, dtype: object

In [136]:
curtailment = (
    result.zonal_net.generators_t.p_max_pu.mul(result.zonal_net.generators.p_nom, axis=1)
    - result.zonal_net.generators_t.p
).clip(lower=0).sum().sum()

In [140]:
result.zonal_net.buses

,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network
Bus,,,,,,,,,,,,,
BE,1.0,,4.897335,50.725288,AC,,,1.0,0.0,inf,PQ,,
DE,1.0,,9.785476,50.923367,AC,,,1.0,0.0,inf,PQ,,
DK-10,1.0,,9.648341,55.892980,AC,,,1.0,0.0,inf,PQ,,
DK-30,1.0,,12.303248,55.515979,AC,,,1.0,0.0,inf,PQ,,
FR,1.0,,3.074425,47.036113,AC,,,1.0,0.0,inf,PQ,,
GB,1.0,,-2.157490,53.228025,AC,,,1.0,0.0,inf,PQ,,
IE,1.0,,-7.591872,52.871207,AC,,,1.0,0.0,inf,PQ,,
LU,1.0,,6.056324,49.684612,AC,,,1.0,0.0,inf,PQ,,
NIR,1.0,,-6.166751,54.640202,AC,,,1.0,0.0,inf,PQ,,


In [131]:
(result.fbmc_parameters["0"].z_ptdf.sel(cnec="0") * result.net_positions).sum()

<xarray.DataArray ()> Size: 8B
array(457139.69691941)
Coordinates:
    branch            <U1 4B '0'
    branch_component  <U4 16B 'Line'
    cnec              <U1 4B '0'

In [132]:
result.zonal_net.lines

,bus0,bus1,type,x,r,g,b,s_nom,s_nom_mod,s_nom_extendable,...,v_ang_max,sub_network,x_pu,r_pu,g_pu,b_pu,x_pu_eff,r_pu_eff,s_nom_opt,v_nom
Line,,,,,,,,,,,,,,,,,,,,,


In [12]:
from fbmc.post_processing.main import process_results

In [13]:
process_results(result, rd_cost= None, rd_dispatch= None, save_path = "/Users/ng-work/Models/pypsa-fbmc/nathan_tests/25-nodes-ep100/results_alt", config = config)

INFO:pypsa.network.io:Exported network 'Unnamed Network'saved to '/Users/ng-work/Models/pypsa-fbmc/nathan_tests/25-nodes-ep100/results_alt/fbmc_network.nc contains: links, storage_units, generators, buses, loads, carriers


{'zonal_market_prices': PosixPath('/Users/ng-work/Models/pypsa-fbmc/nathan_tests/25-nodes-ep100/results_alt/zonal_market_prices.csv'),
 'load_shedding_zone_p': PosixPath('/Users/ng-work/Models/pypsa-fbmc/nathan_tests/25-nodes-ep100/results_alt/load_shedding_zone_p.csv'),
 'generation_mix': PosixPath('/Users/ng-work/Models/pypsa-fbmc/nathan_tests/25-nodes-ep100/results_alt/generation_mix.csv'),
 'storage_mix': PosixPath('/Users/ng-work/Models/pypsa-fbmc/nathan_tests/25-nodes-ep100/results_alt/storage_mix.csv'),
 'storage_units_p': PosixPath('/Users/ng-work/Models/pypsa-fbmc/nathan_tests/25-nodes-ep100/results_alt/storage_units_p.csv'),
 'summary': PosixPath('/Users/ng-work/Models/pypsa-fbmc/nathan_tests/25-nodes-ep100/results_alt/summary.json'),
 'linopy_model': PosixPath('/Users/ng-work/Models/pypsa-fbmc/nathan_tests/25-nodes-ep100/results_alt/linopy_model.nc'),
 'net_positions_zone_p': PosixPath('/Users/ng-work/Models/pypsa-fbmc/nathan_tests/25-nodes-ep100/results_alt/net_positions_zo

In [14]:
result.zonal_net.model.constraints["Bus-nodal_balance"].dual.to_pandas()

KeyError: 'Bus-nodal_balance'

In [76]:
(result.zonal_net.model.constraints["Zone-definition"].loc[:,"BE"].dual.to_pandas() *-1)

snapshot
2013-01-01 00:00:00    13.073799
2013-01-01 01:00:00    13.231006
2013-01-01 02:00:00    14.002359
2013-01-01 03:00:00    14.002543
2013-01-01 04:00:00    13.990512
                         ...    
2013-03-25 03:00:00    27.331982
2013-03-25 04:00:00    27.332003
2013-03-25 05:00:00    27.332059
2013-03-25 06:00:00    27.332022
2013-03-25 07:00:00    27.331950
Length: 2000, dtype: float64

In [21]:
result.zonal_net.statistics.expanded_capacity()

component    carrier                 
Generator    Combined-Cycle Gas          0.0
             Offshore Wind (AC)          0.0
             Offshore Wind (DC)          0.0
             Offshore Wind (Floating)    0.0
             Onshore Wind                0.0
             Open-Cycle Gas              0.0
             Solar                       0.0
             biomass                     0.0
             coal                        0.0
             geothermal                  0.0
             lignite                     0.0
             nuclear                     0.0
             oil                         0.0
             solar-hsat                  0.0
             waste                       0.0
Link         DC                          0.0
StorageUnit  Battery Storage             0.0
dtype: float64

In [128]:
result.zonal_net.model.constraints["Generator-ext-p-upper"]

Constraint `Generator-ext-p-upper` [snapshot: 8736, Generator-ext: 172]:
------------------------------------------------------------------------
[2013-01-01 00:00:00, BE1 0 CCGT]: +1 Generator-p[2013-01-01 00:00:00, BE1 0 CCGT] - 1 Generator-p_nom[BE1 0 CCGT]                  ≤ -0.0
[2013-01-01 00:00:00, BE1 0 nuclear]: +1 Generator-p[2013-01-01 00:00:00, BE1 0 nuclear] - 0.781 Generator-p_nom[BE1 0 nuclear]     ≤ -0.0
[2013-01-01 00:00:00, DE1 0 CCGT]: +1 Generator-p[2013-01-01 00:00:00, DE1 0 CCGT] - 1 Generator-p_nom[DE1 0 CCGT]                  ≤ -0.0
[2013-01-01 00:00:00, DE1 0 OCGT]: +1 Generator-p[2013-01-01 00:00:00, DE1 0 OCGT] - 1 Generator-p_nom[DE1 0 OCGT]                  ≤ -0.0
[2013-01-01 00:00:00, DE1 1 CCGT]: +1 Generator-p[2013-01-01 00:00:00, DE1 1 CCGT] - 1 Generator-p_nom[DE1 1 CCGT]                  ≤ -0.0
[2013-01-01 00:00:00, DE1 1 OCGT]: +1 Generator-p[2013-01-01 00:00:00, DE1 1 OCGT] - 1 Generator-p_nom[DE1 1 OCGT]                  ≤ -0.0
[2013-01-01 00:00:00

In [117]:
nodal_net.statistics.expanded_capacity()

component    carrier                 
Generator    Combined-Cycle Gas               0.89296
             Offshore Wind (AC)               1.34084
             Offshore Wind (DC)               4.58522
             Offshore Wind (Floating)         3.49939
             Onshore Wind                 38837.28688
             Open-Cycle Gas                   0.10507
             Solar                        19085.71321
             biomass                          0.00000
             coal                             0.00000
             geothermal                       0.00000
             lignite                          0.00000
             nuclear                          0.42779
             oil                              0.00000
             solar-hsat                  135482.99670
             waste                            0.00000
Line         AC                           85080.20262
Link         DC                              14.58120
StorageUnit  Battery Storage              26

In [118]:
result.zonal_net.statistics.expanded_capacity()

component    carrier                 
Generator    Combined-Cycle Gas               0.89296
             Offshore Wind (AC)               1.34084
             Offshore Wind (DC)               4.58522
             Offshore Wind (Floating)         3.49939
             Onshore Wind                 38837.28688
             Open-Cycle Gas                   0.10507
             Solar                        19085.71321
             biomass                          0.00000
             coal                             0.00000
             geothermal                       0.00000
             lignite                          0.00000
             nuclear                          0.42779
             oil                              0.00000
             solar-hsat                  135482.99670
             waste                            0.00000
Link         DC                               3.68863
StorageUnit  Battery Storage              26355.85776
dtype: float64